# Tachiom Python API Demo

Covers the full `tachiom` Python interface:
- **Build** from raw `.npy` files (full pipeline or from pre-computed TAC)
- **Load** a previously saved index
- **Inspect** index properties
- **Search** — single query and batch
- **Evaluate** with `ir_measures`
- **Grid sweep** over retrieval parameters

Input files (LOTTE, `bench_tachiom_lotte_two_levels_pq.toml`):

| File | Shape | dtype | Description |
|---|---|---|---|
| `documents.npy` | `[N, dim]` | `f16` | Token vectors |
| `document_token_ids_flat.npy` | `[N]` | `i64`/`u32` | Token-type ID per token |
| `doclens.npy` | `[n_docs]` | `i32`/`i64` | Tokens per document |
| `queries.npy` | `[Q, n_tok, dim]` | `f32` | Query token vectors |
| `documents_ids.npy` *(opt)* | `[n_docs]` | any | Integer index → string doc ID |
| `queries_ids.npy` *(opt)* | `[Q]` | any | Integer index → string query ID |
| `centroids.npy` *(opt, TAC)* | `[K, dim]` | `f32` | Pre-computed coarse centroids |
| `assignments.npy` *(opt, TAC)* | `[N]` | `u32`/`u64` | Centroid assignment per token |

---
## 0b — Build the shared library

Run once per code change. `target-cpu=native` enables SIMD (AVX2/AVX-512) for PQ kernels.

In [1]:
import subprocess, sys, os

result = subprocess.run(
    ["maturin", "develop", "--release"],
    cwd="..",
    env={**os.environ, "RUSTFLAGS": "-C target-cpu=native"},
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("maturin build failed")
print("Build OK")

✏️ Setting installed package as editable

Build OK


---
## 1 — Configuration

In [ ]:
from pathlib import Path

# ── Paths — update these to match your local setup ───────────────────────────
DATA_DIR   = Path("/path/to/dataset")
INDEX_PATH = Path("/path/to/indexes/tachiom/my_index")
QRELS_PATH = Path("/path/to/qrels.tsv")

VECTORS_FILE   = DATA_DIR / "documents.npy"
TOKEN_IDS_FILE = DATA_DIR / "document_token_ids_flat.npy"
DOCLENS_FILE   = DATA_DIR / "doclens.npy"
QUERIES_FILE   = DATA_DIR / "queries.npy"
DOC_IDS_FILE   = DATA_DIR / "documents_ids.npy"   # set to None to skip
QUERY_IDS_FILE = DATA_DIR / "queries_ids.npy"     # set to None to skip

# Only needed for Option B (build_from_tac)
CENTROIDS_FILE   = DATA_DIR / "centroids.npy"     # [K, dim] f32
ASSIGNMENTS_FILE = DATA_DIR / "assignments.npy"   # [N] u32/u64

# ── Build params ──────────────────────────────────────────────────────────────
BUILD_PARAMS = dict(
    total_centroids = 2_097_152,
    tac_n_iter      = 10,
    pq_sample_size  = 10_000_000,
    pq_n_iter       = 10,
    normalize       = True,
    pq_seed         = 42,
    hnsw_m          = 32,
    ef_construction = 1500,
)

# ── Search params ─────────────────────────────────────────────────────────────
K             = 10
SEARCH_PARAMS = dict(
    k_centroids     = 20,
    k_docs_to_score = 200,
    ef_search       = 30,
    alpha           = 0.4,
    beta            = None,
    lambda_         = None,
    num_threads     = 0,   # 0 = all cores; 1 = serial
)

EVAL_METRIC = "Success@5"

print("Config OK")

---
## 2 — Build / Load the index

Run **exactly one** of the three cells below.

### Option A — Build from scratch (full TAC → PQ → HNSW pipeline)

Use when you have raw token vectors and no pre-computed centroids.

In [ ]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.build(
    str(VECTORS_FILE),
    str(TOKEN_IDS_FILE),
    str(DOCLENS_FILE),
    **BUILD_PARAMS,
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
idx.save(str(INDEX_PATH))
print(f"Saved to {INDEX_PATH}")

### Option B — Build from pre-computed TAC centroids (skip clustering)

Use when you already have `centroids.npy` and `assignments.npy` from a previous TAC run.  
Runs PQ training + encoding from scratch but skips the k-means step.

In [ ]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.build_from_tac(
    str(VECTORS_FILE),
    str(TOKEN_IDS_FILE),
    str(DOCLENS_FILE),
    str(CENTROIDS_FILE),
    str(ASSIGNMENTS_FILE),
    pq_sample_size  = BUILD_PARAMS["pq_sample_size"],
    pq_n_iter       = BUILD_PARAMS["pq_n_iter"],
    normalize       = BUILD_PARAMS["normalize"],
    pq_seed         = BUILD_PARAMS["pq_seed"],
    hnsw_m          = BUILD_PARAMS["hnsw_m"],
    ef_construction = BUILD_PARAMS["ef_construction"],
)
print(f"Built in {time.perf_counter() - t0:.2f}s")

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
idx.save(str(INDEX_PATH))
print(f"Saved to {INDEX_PATH}")

### Option C — Load a previously saved index

In [3]:
import time, tachiom

t0 = time.perf_counter()
idx = tachiom.Tachiom.load(str(INDEX_PATH))
print(f"Loaded in {time.perf_counter() - t0:.2f}s")

Loaded in 59.42s


---
## 3 — Index inspection

Properties: `len`, `dim`, `n_tokens`, `n_centroids`.  
Method: `print_space_usage()`.

In [4]:
print(repr(idx))
print(f"Documents  : {idx.len:,}")
print(f"Dim        : {idx.dim}")
print(f"Tokens     : {idx.n_tokens:,}")
print(f"Centroids  : {idx.n_centroids:,}")
print()
idx.print_space_usage()

<Tachiom: 2428854 docs, dim=128, 2097152 centroids>
Documents  : 2,428,854
Dim        : 128
Tokens     : 266,205,513
Centroids  : 2,097,152

Index space usage:
  centroids_hnsw         0.76 GB  (  6.2%)
  inverted_lists         1.00 GB  (  8.2%)
  offsets                0.03 GB  (  0.3%)
  residuals             10.44 GB  ( 85.3%)
  ──────────────────────────────────────
  total                 12.23 GB


---
## 4 — Load queries

In [5]:
import numpy as np

# shape (Q, n_tokens, dim), must be float32 C-contiguous
queries = np.ascontiguousarray(np.load(QUERIES_FILE).astype(np.float32))
assert queries.ndim == 3, f"Expected (Q, n_tokens, dim), got {queries.shape}"
assert queries.shape[2] == idx.dim, f"dim mismatch: {queries.shape[2]} vs {idx.dim}"

doc_ids_arr   = np.load(DOC_IDS_FILE,   allow_pickle=True) if DOC_IDS_FILE   and DOC_IDS_FILE.exists()   else None
query_ids_arr = np.load(QUERY_IDS_FILE, allow_pickle=True) if QUERY_IDS_FILE and QUERY_IDS_FILE.exists() else None

print(f"Queries : {queries.shape}  (dtype={queries.dtype})")
print(f"doc_ids loaded   : {doc_ids_arr is not None}")
print(f"query_ids loaded : {query_ids_arr is not None}")

Queries : (2931, 32, 128)  (dtype=float32)
doc_ids loaded   : True
query_ids loaded : True


---
## 5 — Single-query search

`idx.search(query, k, *, k_centroids, k_docs_to_score, ef_search, alpha, beta, lambda_)`  
Input: 2D `f32` array `(n_tokens, dim)`. Returns `(scores, doc_ids)` — both 1D, length `k`.

In [6]:
import time

q = np.ascontiguousarray(queries[0])  # (n_tokens, dim)

t0 = time.perf_counter()
sc, di = idx.search(
    q, k=K,
    k_centroids     = SEARCH_PARAMS["k_centroids"],
    k_docs_to_score = SEARCH_PARAMS["k_docs_to_score"],
    ef_search       = SEARCH_PARAMS["ef_search"],
    alpha           = SEARCH_PARAMS["alpha"],
    beta            = SEARCH_PARAMS["beta"],
    lambda_         = SEARCH_PARAMS["lambda_"],
)
print(f"Latency: {(time.perf_counter() - t0)*1000:.1f} ms")
print("scores: [", end="")
for score in sc[:-1]:
    print(f"{score:.4f}, ", end="")
print(f"{sc[-1]:.4f}]")

print(f"doc_ids: [", end="")
for d_idx in di[:-1]:
    print(f"{d_idx}, ", end="")
print(f"{di[-1]}]")
print()

sentinel = np.iinfo(np.uint32).max
print("Top results for query 0:")
for rank, (score, d_idx) in enumerate(zip(sc, di), 1):
    if d_idx == sentinel:
        break
    label = str(doc_ids_arr[d_idx]) if doc_ids_arr is not None else str(d_idx)
    print(f"  {rank:2d}.  doc={label}  score={score:.4f}")

Latency: 34.8 ms
scores: [23.1674, 21.1828, 21.1480, 21.0740, 20.9822, 20.6083, 20.6065, 20.3865, 20.3649, 20.3587]
doc_ids: [41780, 14612, 107776, 8893, 54678, 44093, 38079, 77013, 38056, 778]

Top results for query 0:
   1.  doc=41780  score=23.1674
   2.  doc=14612  score=21.1828
   3.  doc=107776  score=21.1480
   4.  doc=8893  score=21.0740
   5.  doc=54678  score=20.9822
   6.  doc=44093  score=20.6083
   7.  doc=38079  score=20.6065
   8.  doc=77013  score=20.3865
   9.  doc=38056  score=20.3649
  10.  doc=778  score=20.3587


---
## 6 — Batch search

`idx.batch_search(queries, k, *, num_threads, k_centroids, k_docs_to_score, ef_search, alpha, beta, lambda_)`  
Input: 3D `f32` array `(Q, n_tokens, dim)`. Returns `(scores, doc_ids)` — both 2D `(Q, k)`.

In [7]:
import time

n_queries = len(queries)
t0 = time.perf_counter()
scores, doc_indices = idx.batch_search(queries, k=K, **SEARCH_PARAMS)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Total : {elapsed_ms:.1f} ms   per query : {elapsed_ms / n_queries:.2f} ms")
print(f"scores     shape : {scores.shape}     dtype={scores.dtype}")
print(f"doc_indices shape : {doc_indices.shape}  dtype={doc_indices.dtype}")

Total : 1645.3 ms   per query : 0.56 ms
scores     shape : (2931, 10)     dtype=float32
doc_indices shape : (2931, 10)  dtype=uint32


---
## 7 — Evaluate with ir_measures

Builds a TREC run from the batch-search output and computes the configured metric.

In [8]:
import pandas as pd, ir_measures

sentinel = np.iinfo(np.uint32).max
rows = []
for q_idx in range(n_queries):
    q_id = str(query_ids_arr[q_idx]) if query_ids_arr is not None else str(q_idx)
    for rank, (s, d) in enumerate(zip(scores[q_idx], doc_indices[q_idx]), 1):
        if d == sentinel:
            break
        d_id = str(doc_ids_arr[d]) if doc_ids_arr is not None else str(d)
        rows.append({"query_id": q_id, "doc_id": d_id, "rank": rank, "score": float(s)})

run_df = pd.DataFrame(rows)
print(f"{len(run_df)} rows in run")
run_df.head()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File 

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/smartinico/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



29310 rows in run


,query_id,doc_id,rank,score
0,0,41780,1,23.167360
1,0,14612,2,21.182775
2,0,107776,3,21.147970
3,0,8893,4,21.074039
4,0,54678,5,20.982195


In [9]:
qrels_df = pd.read_csv(QRELS_PATH, sep="\t",
                       names=["query_id", "useless", "doc_id", "relevance"])
if len(pd.unique(qrels_df["useless"])) != 1:  # 3-col format
    qrels_df = pd.read_csv(QRELS_PATH, sep="\t",
                           names=["query_id", "doc_id", "relevance", "useless"])

for df in (qrels_df, run_df):
    df["query_id"] = df["query_id"].astype(str)
    df["doc_id"]   = df["doc_id"].astype(str)

print(f"{len(qrels_df)} qrels rows")
qrels_df.head()

8573 qrels rows


,query_id,useless,doc_id,relevance
0,0,Q0,9032,1
1,1,Q0,39506,1
2,1,Q0,39507,1
3,2,Q0,4893,1
4,2,Q0,4895,1


In [10]:
metric = ir_measures.parse_measure(EVAL_METRIC)
result = ir_measures.calc_aggregate([metric], qrels_df, run_df)
print(f"{metric}: {result[metric]:.6f}")

Success@5: 0.675879


---
## 8 — Parameter grid sweep

Three configs from the TOML (expected values are from the TOML comments — results may vary slightly due to RNG).

In [11]:
import time, pandas as pd, ir_measures

metric = ir_measures.parse_measure(EVAL_METRIC)
sentinel = np.iinfo(np.uint32).max

configs = [
    dict(ef_search=30, k_centroids=20, k_docs_to_score=200, alpha=0.4),  # ≈ 67.5, 11 ms/q sequentially
    dict(ef_search=30, k_centroids=20, k_docs_to_score=300, alpha=0.4),  # ≈ 68.0, 14 ms/q sequentially
    dict(ef_search=40, k_centroids=25, k_docs_to_score=500, alpha=0.4),  # ≈ 68.5, 23 ms/q sequentially
]

rows_out = []
for cfg in configs:
    t0 = time.perf_counter()
    sc, di = idx.batch_search(queries, k=K, num_threads=SEARCH_PARAMS["num_threads"], **cfg)
    ms_per_q = (time.perf_counter() - t0) * 1000 / n_queries

    rows_run = []
    for q_idx in range(n_queries):
        q_id = str(query_ids_arr[q_idx]) if query_ids_arr is not None else str(q_idx)
        for rank, (s, d) in enumerate(zip(sc[q_idx], di[q_idx]), 1):
            if d == sentinel:
                break
            d_id = str(doc_ids_arr[d]) if doc_ids_arr is not None else str(d)
            rows_run.append({"query_id": q_id, "doc_id": d_id, "rank": rank, "score": float(s)})
    run_cfg = pd.DataFrame(rows_run)
    run_cfg["query_id"] = run_cfg["query_id"].astype(str)
    run_cfg["doc_id"]   = run_cfg["doc_id"].astype(str)

    val = ir_measures.calc_aggregate([metric], qrels_df, run_cfg)[metric]
    rows_out.append({**cfg, EVAL_METRIC: round(val, 4), "ms/query": round(ms_per_q, 2)})
    print(f"{cfg}  {EVAL_METRIC}={val:.4f}  {ms_per_q:.2f} ms/q")

pd.DataFrame(rows_out)

{'ef_search': 30, 'k_centroids': 20, 'k_docs_to_score': 200, 'alpha': 0.4}  Success@5=0.6759  0.53 ms/q
{'ef_search': 30, 'k_centroids': 20, 'k_docs_to_score': 300, 'alpha': 0.4}  Success@5=0.6796  0.63 ms/q
{'ef_search': 40, 'k_centroids': 25, 'k_docs_to_score': 500, 'alpha': 0.4}  Success@5=0.6847  0.85 ms/q


,ef_search,k_centroids,k_docs_to_score,alpha,Success@5,ms/query
0,30,20,200,0.4,0.6759,0.53
1,30,20,300,0.4,0.6796,0.63
2,40,25,500,0.4,0.6847,0.85
